In [2]:
%pip install pandas scikit-learn matplotlib tqdm

  Using cached pandas-2.2.3-cp312-cp312-win_amd64.whl.metadata (19 kB)
  Using cached pytz-2024.2-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tzdata-2024.2-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached joblib-1.4.2-py3-none-any.whl.metadata (5.4 kB)
  Using cached threadpoolctl-3.5.0-py3-none-any.whl.metadata (13 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
Using cached pandas-2.2.3-cp312-cp312-win_amd64.whl (11.5 MB)
   ---------------------------------------- 0.0/11.1 MB ? eta -:--:--
    --------------------------------------- 0.3/11.1 MB ? eta -:--:--
   ---- ----------------------------------- 1.3/11.1 MB 3.7 MB/s eta 0:00:03
   ------- -------------------------------- 2.1/11.1 MB 3.8 MB/s eta 0:00:03
   ---------- ----------------------------- 2.9/11.1 MB 3.7 MB/s eta 0:00:03
   ------------- -------------------------- 3.7/11.1 MB 3.8 MB/s eta 0:00:02
   ---------------- ----------------------- 4.7/11.1 MB 3.9 MB/s eta 0:00:02
   ----------

In [ ]:
import tensorflow as tf
from keras.applications.inception_v3 import InceptionV3, preprocess_input
from keras.layers import (
    GlobalAveragePooling2D,
    Dense,
    Dropout,
    Input,
    Conv2D,
    multiply,
    Lambda,
    BatchNormalization,
)
from keras.models import Model
import numpy as np

IMG_SIZE = (512, 512)
in_lay = Input(shape=(*IMG_SIZE, 3))
base_pretrained_model = InceptionV3(
    input_shape=(*IMG_SIZE, 3), include_top=False, weights="imagenet"
)
base_pretrained_model.trainable = False

pt_depth = base_pretrained_model.output_shape[-1]
pt_features = base_pretrained_model(in_lay)
bn_features = BatchNormalization()(pt_features)

attn_layer = Conv2D(64, kernel_size=(1, 1), padding="same", activation="relu")(
    Dropout(0.5)(bn_features)
)
attn_layer = Conv2D(16, kernel_size=(1, 1), padding="same", activation="relu")(
    attn_layer
)
attn_layer = Conv2D(8, kernel_size=(1, 1), padding="same", activation="relu")(
    attn_layer
)
attn_layer = Conv2D(1, kernel_size=(1, 1), padding="valid", activation="sigmoid")(
    attn_layer
)

up_c2_w = np.ones((1, 1, 1, pt_depth))
up_c2 = Conv2D(
    pt_depth,
    kernel_size=(1, 1),
    padding="same",
    activation="linear",
    use_bias=False,
    weights=[up_c2_w],
)
up_c2.trainable = False
attn_layer = up_c2(attn_layer)

mask_features = multiply([attn_layer, bn_features])
gap_features = GlobalAveragePooling2D()(mask_features)
gap_mask = GlobalAveragePooling2D()(attn_layer)
gap = Lambda(lambda x: x[0] / x[1], name="RescaleGAP")([gap_features, gap_mask])
gap_dr = Dropout(0.25)(gap)
dr_steps = Dropout(0.25)(Dense(128, activation="relu")(gap_dr))
out_layer = Dense(5, activation="softmax")(dr_steps)
retina_model = Model(inputs=[in_lay], outputs=[out_layer])

# Compile the model
from keras.metrics import top_k_categorical_accuracy


def top_2_accuracy(in_gt, in_pred):
    return top_k_categorical_accuracy(in_gt, in_pred, k=2)


retina_model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["categorical_accuracy", top_2_accuracy],
)

# Load the weights
retina_model.load_weights("path/to/your/retina_weights.best.hdf5")

ValueError: Layer count mismatch when loading weights from file. Model expected 9 layers, found 243 saved layers.